In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


In [6]:
# get all csv filepaths as aboslute paths
data_csvs = list(map(Path.resolve, Path('../data/').rglob("*.csv")))
for fp in data_csvs:
    print(fp)

/Users/yung/repos/DSC445-ML1-Election-Prediction/data/Raw/countypres_2000-2024.csv
/Users/yung/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B23025-Data.csv
/Users/yung/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B19013-Data.csv
/Users/yung/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B25001-Data.csv
/Users/yung/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B17001-Data.csv
/Users/yung/repos/DSC445-ML1-Election-Prediction/data/Raw/demographic/ACSDT5Y2020.B02001-Data.csv
/Users/yung/repos/DSC445-ML1-Election-Prediction/data/Raw/demographic/ACSDT5Y2020.B15003-Data.csv
/Users/yung/repos/DSC445-ML1-Election-Prediction/data/Raw/demographic/ACSDT5Y2020.B01001-Data.csv


## Election Dataset

Column Summary

| Columns        | Description                                |
| -------------- | ------------------------------------------ |
| state          | Full state name                            |
| state_po       | State abbreviation                         |
| county_name    | County name                                |
| county_fips    | County FIPS code used for merging datasets |
| year           | Election year                              |
| office         | Election office (US PRESIDENT)             |
| candidate      | Candidate name                             |
| party          | Political party                            |
| candidatevotes | Votes received by candidate                |
| totalvotes     | Total votes in county                      |

Column Status:

- State/State_PO: duplicate data set
- county_name: good info, but provide duplicate data to county_fips, also includes state data
- county_fips: has missing data, but can be addressed easily with dummy data
- year: used for splitting dataset by year
- office: only contains 'president', not needed
- candidate name: not needed as the prediction is done by party but is a good reference
- candidate votes/total votes: self explanatory, good continuous values

Overall, 

This dataset will be **split** along `year`, provides **label** through `candidate` or `party`, and `county_fips`, `candidatevotes`, `totalvotes` as **features**. 




In [12]:
election_csv = data_csvs[0]
# import as strings
election_df = pd.read_csv(election_csv, dtype={'county_fips': str})

## Basic Data Check

In [19]:
election_df.head()

,state,county_name,year,state_po,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,OTHER,OTHER,293.0,28281,20260225,TOTAL
1,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,CHASE OLIVER,LIBERTARIAN,65.0,28281,20260225,TOTAL
2,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,KAMALA D HARRIS,DEMOCRAT,7439.0,28281,20260225,TOTAL
3,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,20484.0,28281,20260225,TOTAL
4,ALABAMA,BALDWIN,2024,AL,1003,US PRESIDENT,OTHER,OTHER,1276.0,122249,20260225,TOTAL


In [78]:
election_df.office.unique()

<StringArray>
['US PRESIDENT']
Length: 1, dtype: str

### Review nan Values

In [23]:
nan_columns = []
for col in election_df.columns:
    nan_count = election_df[col].isna().sum()
    if nan_count:
        print(f'{col} np.nan count: {nan_count}')
        nan_columns.append(col)

county_fips np.nan count: 52
party np.nan count: 501
candidatevotes np.nan count: 37
mode np.nan count: 2795


In [29]:
nan_rows_per_columns = {
    n_col: election_df.index[election_df[n_col].isna()]
    for n_col in nan_columns
}

#### Missing County Fips

Missing County Fips for these locations can be addressed by adding dummy FIPS codes such as 99001, 99002, 99003, etc

In [74]:
def get_missing_sets(indices: pd.Index,
                     set_columns: list,
                     df: pd.DataFrame=election_df) -> set:
    """helper for getting a set of values to check for missing values"""
    _df = df.loc[indices, set_columns]
    return _df.drop_duplicates().values

In [75]:
missing_fips = get_missing_sets(nan_rows_per_columns['county_fips'],
                                ('state', 'county_name'))
print(missing_fips)

[['CONNECTICUT' 'STATEWIDE WRITEIN']
 ['MAINE' 'MAINE UOCAVA']
 ['RHODE ISLAND' 'FEDERAL PRECINCT']]


In [100]:
missing_party = get_missing_sets(nan_rows_per_columns['candidatevotes'],
                                ('state', 'candidate', 'party'))
print(missing_party)

[['NEW MEXICO' 'CHASE OLIVER' 'LIBERTARIAN']]


## Socioeconomic Dataset

first is geographic location

In [ ]:
se_fps = data_csvs[1:5]
se_column_info = {}
for fp in se_fps:
    _df = pd.read_csv(fp)
    se_column_info[fp.name] = _df.iloc[0, :].to_dict()

In [96]:
se_column_info

{'ACSDT5Y2020.B23025-Data.csv': {'GEO_ID': 'Geography',
  'NAME': 'Geographic Area Name',
  'B23025_001E': 'Estimate!!Total:',
  'B23025_001M': 'Margin of Error!!Total:',
  'B23025_002E': 'Estimate!!Total:!!In labor force:',
  'B23025_002M': 'Margin of Error!!Total:!!In labor force:',
  'B23025_003E': 'Estimate!!Total:!!In labor force:!!Civilian labor force:',
  'B23025_003M': 'Margin of Error!!Total:!!In labor force:!!Civilian labor force:',
  'B23025_004E': 'Estimate!!Total:!!In labor force:!!Civilian labor force:!!Employed',
  'B23025_004M': 'Margin of Error!!Total:!!In labor force:!!Civilian labor force:!!Employed',
  'B23025_005E': 'Estimate!!Total:!!In labor force:!!Civilian labor force:!!Unemployed',
  'B23025_005M': 'Margin of Error!!Total:!!In labor force:!!Civilian labor force:!!Unemployed',
  'B23025_006E': 'Estimate!!Total:!!In labor force:!!Armed Forces',
  'B23025_006M': 'Margin of Error!!Total:!!In labor force:!!Armed Forces',
  'B23025_007E': 'Estimate!!Total:!!Not in l

## Demo Data
- divides demo graphic

In [97]:
dem_fps = data_csvs[5:]
dem_column_info = {}
for fp in dem_fps:
    _df = pd.read_csv(fp)
    dem_column_info[fp.name] = _df.iloc[0, :].to_dict()

In [98]:
dem_column_info

{'ACSDT5Y2020.B02001-Data.csv': {'GEO_ID': 'Geography',
  'NAME': 'Geographic Area Name',
  'B02001_001E': 'Estimate!!Total:',
  'B02001_001M': 'Margin of Error!!Total:',
  'B02001_002E': 'Estimate!!Total:!!White alone',
  'B02001_002M': 'Margin of Error!!Total:!!White alone',
  'B02001_003E': 'Estimate!!Total:!!Black or African American alone',
  'B02001_003M': 'Margin of Error!!Total:!!Black or African American alone',
  'B02001_004E': 'Estimate!!Total:!!American Indian and Alaska Native alone',
  'B02001_004M': 'Margin of Error!!Total:!!American Indian and Alaska Native alone',
  'B02001_005E': 'Estimate!!Total:!!Asian alone',
  'B02001_005M': 'Margin of Error!!Total:!!Asian alone',
  'B02001_006E': 'Estimate!!Total:!!Native Hawaiian and Other Pacific Islander alone',
  'B02001_006M': 'Margin of Error!!Total:!!Native Hawaiian and Other Pacific Islander alone',
  'B02001_007E': 'Estimate!!Total:!!Some other race alone',
  'B02001_007M': 'Margin of Error!!Total:!!Some other race alone

- my goals
- i do linear
- atif does lcassifcation

Make the below as the project goal

- split by year where 2024 is the prediction data set for all models
- 2020 and previous is the data set

once the data is merged and cleaned, people get data sets and do prediction on different years for their model
